# PlanForge
> Your AI business advisor. Tell it what you want to achieve, have a conversation, get a plan.

*v0.5.0*

In [ ]:
# @title Setup
# @markdown Run this cell every session.
# @markdown
# @markdown First time? [Getting started guide](https://planforge.pysolvr.com/docs/getting-started)
# @markdown - Save a copy: File > Save a copy in Drive
# @markdown - Add API key: click key icon (left sidebar) > add `PLANFORGE_API_KEY`
!pip install -q -U pysolvr-client
import sys
from google.colab import userdata, drive

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    from IPython.display import HTML, display
    display(HTML('<div style="background:#1e293b;border:1px solid #ef4444;border-radius:8px;padding:16px;font-family:Inter,system-ui,sans-serif;color:#f1f5f9"><b style="color:#ef4444">API key not found in Colab Secrets</b><ol style="color:#94a3b8;font-size:13px;margin-top:8px"><li>Click the key icon in the left sidebar</li><li>Add secret: <code>PLANFORGE_API_KEY</code></li><li>Paste your API key as the value</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'API key not configured - see instructions above'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev/'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'exports', 'uploads'])
drive_mgr.ensure_folders()

if client.health_check():
    ui.success('Connected to PlanForge API', f'Drive: {drive_mgr.root}')
else:
    ui.error('Could not connect to API', 'Check your API key and try again')
import ipywidgets as widgets
import time as _time
from IPython.display import clear_output
_state = {'goal': None, 'goal_info': {}, 'topics': [], 'current_topic': None, 'history': {}, 'locked': {}, 'profile': {}}


In [ ]:
# @title Profile
# @markdown Pro tier only. Starter accounts can skip this cell -- your default profile is set automatically.

_prof_main = widgets.Output()
display(_prof_main)

_DEFAULT_PROFILE = {'id': 'default', 'name': 'Default', 'facts': {}, 'locked_topics': {}}


def _save_profile_to_drive(profile):
    drive_mgr.save('profiles', f"{profile['id']}.json", '', meta=profile)


def _load_profiles():
    files = drive_mgr.list_files('profiles')
    profiles = [f['meta'] for f in files if f['name'].endswith('.json') and f.get('meta')]
    if not profiles:
        _save_profile_to_drive(_DEFAULT_PROFILE)
        profiles = [_DEFAULT_PROFILE]
    return profiles


def _activate_profile(profile):
    _state['profile'] = profile
    facts = dict(profile.get('facts', {}))
    for k in ('name', 'industry', 'stage', 'description'):
        if profile.get(k) and k not in facts:
            facts[k] = profile[k]
    _state['profile']['facts'] = facts


def _render_starter():
    _faq_out = widgets.Output()
    _faq_visible = [False]
    _faq_btn = widgets.Button(
        description="What's this?",
        button_style='',
        layout=widgets.Layout(width='100px', height='28px'),
    )

    def _toggle_faq(b):
        _faq_visible[0] = not _faq_visible[0]
        with _faq_out:
            clear_output(wait=True)
            if _faq_visible[0]:
                ui.card(
                    'Profiles',
                    'Profiles let you save your business context once and reuse it across plans. '
                    'When you answer discovery questions, those answers are locked to your profile -- '
                    'every new plan starts with that context already filled in. '
                    'Pro users can create multiple profiles, one per business, client, or persona.',
                )

    _faq_btn.on_click(_toggle_faq)

    _name_in = widgets.Text(placeholder='Business or your name (optional)',
                             layout=widgets.Layout(width='260px'))
    _desc_in = widgets.Textarea(placeholder='One line about your business (optional) -- helps the AI tailor suggestions',
                                layout=widgets.Layout(width='100%', height='56px'))
    _save_btn = widgets.Button(description='Save', button_style='primary',
                               layout=widgets.Layout(width='70px', height='32px'))
    _save_out2 = widgets.Output()

    def _on_save_starter(b):
        facts = {}
        if _name_in.value.strip(): facts['name'] = _name_in.value.strip()
        if _desc_in.value.strip(): facts['description'] = _desc_in.value.strip()
        _state['profile']['facts'] = facts
        drive_mgr.save('profiles', 'default.json', '', meta=_state['profile'])
        with _save_out2:
            clear_output(wait=True)
            ui.success('Saved', 'Context will be used in all your plans.')

    _save_btn.on_click(_on_save_starter)

    _upsell = widgets.HTML(
        '<p style="color:#64748b;font-size:12px;margin:8px 0 0">'
        'Need plans for multiple businesses or clients? '
        '<a href="https://planforge.pysolvr.com/#pricing" target="_blank" style="color:#3b82f6">'
        'Upgrade to Pro</a></p>'
    )

    with _prof_main:
        clear_output(wait=True)
        display(ui.card_widget([
            widgets.HBox([_faq_btn], layout=widgets.Layout(align_items='center')),
            _faq_out,
            widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:8px 0 4px">Tell the AI about you (optional)</p>'),
            _name_in, _desc_in,
            widgets.HBox([_save_btn], layout=widgets.Layout(margin='6px 0 0')),
            _save_out2,
            _upsell,
        ], title='Profile'))
        _saved = drive_mgr.list_files('profiles')
        _saved_meta = next((f['meta'] for f in _saved if f['name'] == 'default.json' and f.get('meta')), None)
        _activate_profile(_saved_meta if _saved_meta else _DEFAULT_PROFILE)


def _render_pro(profiles):
    _status_out = widgets.Output()
    _form_out = widgets.Output()

    active_id = _state.get('profile', {}).get('id', 'default')
    options = [(p['name'], p['id']) for p in profiles]
    active_idx = next((i for i, (_, pid) in enumerate(options) if pid == active_id), 0)

    _selector = widgets.Dropdown(
        options=options,
        index=active_idx,
        layout=widgets.Layout(width='220px'),
    )
    _new_btn = widgets.Button(
        description='+ New Profile',
        button_style='primary',
        layout=widgets.Layout(width='120px', height='32px'),
    )

    _profiles_by_id = {p['id']: p for p in profiles}

    def _on_select(change):
        pid = change['new']
        profile = _profiles_by_id.get(pid, _DEFAULT_PROFILE)
        _activate_profile(profile)
        with _status_out:
            clear_output(wait=True)
            ui.success(
                f"Profile: {profile['name']}",
                f"{len(profile.get('locked_topics', {}))} topics loaded",
            )
        with _form_out:
            clear_output(wait=True)

    _selector.observe(_on_select, names='value')

    def _on_new(btn):
        with _form_out:
            clear_output(wait=True)
            _name = widgets.Text(placeholder='Profile name (required)',
                                  layout=widgets.Layout(width='220px'))
            _industry = widgets.Text(placeholder='Industry (e.g. SaaS, Retail)',
                                      layout=widgets.Layout(width='220px'))
            _stage = widgets.Dropdown(
                options=['', 'Pre-idea', 'Idea', 'MVP', 'Early revenue', 'Growth', 'Scale'],
                description='Stage:',
                style={'description_width': '50px'},
                layout=widgets.Layout(width='220px'),
            )
            _desc = widgets.Textarea(
                placeholder='Brief description (optional) -- anything the AI should know',
                layout=widgets.Layout(width='100%', height='60px'),
            )
            _create_btn = widgets.Button(description='Create', button_style='success',
                                          layout=widgets.Layout(width='90px', height='32px'))
            _cancel_btn = widgets.Button(description='Cancel', button_style='',
                                          layout=widgets.Layout(width='80px', height='32px'))
            _form_status = widgets.Output()

            def _on_create(b):
                name = _name.value.strip()
                if not name:
                    with _form_status:
                        clear_output(wait=True)
                        ui.error('Profile name is required')
                    return
                pid = name.lower().replace(' ', '-').replace('/', '-')[:40]
                new_profile = {
                    'id': pid, 'name': name,
                    'facts': {k: v for k, v in {
                        'name': name,
                        'industry': _industry.value.strip(),
                        'stage': _stage.value,
                        'description': _desc.value.strip(),
                    }.items() if v},
                    'locked_topics': {},
                }
                _save_profile_to_drive(new_profile)
                _profiles_by_id[pid] = new_profile
                _activate_profile(new_profile)
                _selector.options = list(_selector.options) + [(name, pid)]
                _selector.value = pid
                with _form_out:
                    clear_output(wait=True)
                with _status_out:
                    clear_output(wait=True)
                    ui.success(
                        f'Profile created: {name}',
                        'Start discovery to build its context.',
                    )

            def _on_cancel(b):
                with _form_out:
                    clear_output(wait=True)

            _create_btn.on_click(_on_create)
            _cancel_btn.on_click(_on_cancel)
            display(widgets.VBox([
                widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:0 0 8px">New profile</p>'),
                widgets.HTML('<p style="color:#64748b;font-size:11px;margin:0 0 8px">Name is required. Other fields help the AI tailor plans.</p>'),
                _name, _industry, _stage, _desc,
                widgets.HBox([_create_btn, _cancel_btn], layout=widgets.Layout(gap='8px', margin='8px 0 0')),
                _form_status,
            ]))

    _new_btn.on_click(_on_new)

    with _prof_main:
        clear_output(wait=True)
        display(ui.card_widget([
            widgets.HBox([_selector, _new_btn], layout=widgets.Layout(align_items='center', gap='10px')),
            _status_out,
            _form_out,
        ], title='Profile'))


# --- entry point ---
_acct = client.call('GET', '/account')
_tier = _acct['data'].get('tier', 'starter') if _acct.get('ok') else 'starter'

if _tier == 'pro':
    _profiles = _load_profiles()
    _activate_profile(_profiles[0])
    _render_pro(_profiles)
else:
    _render_starter()


In [ ]:
# @title My plans

# ── helpers ──────────────────────────────────────────────────────────────
import re as _re

def _md_to_html(text):
    lines = []
    in_ul = False
    for line in text.split('\n'):
        if line.startswith('### '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h4 style="color:#f1f5f9;margin:16px 0 4px;font-size:14px">{line[4:]}</h4>')
        elif line.startswith('## '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h3 style="color:#f1f5f9;margin:20px 0 6px;font-size:16px;border-bottom:1px solid #475569;padding-bottom:4px">{line[3:]}</h3>')
        elif line.startswith('# '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h2 style="color:#e2e8f0;margin:24px 0 8px;font-size:18px">{line[2:]}</h2>')
        elif line.startswith('- ') or line.startswith('* '):
            if not in_ul: lines.append('<ul style="margin:4px 0 4px 16px;padding:0;color:#cbd5e1;font-size:13px">'); in_ul = True
            content = _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', line[2:])
            lines.append(f'<li style="margin:2px 0">{content}</li>')
        else:
            if in_ul: lines.append('</ul>'); in_ul = False
            if line.strip():
                content = _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', line)
                lines.append(f'<p style="color:#cbd5e1;font-size:13px;margin:4px 0;line-height:1.6">{content}</p>')
    if in_ul: lines.append('</ul>')
    return '\n'.join(lines)


# ── state ────────────────────────────────────────────────────────────────
_selected = [None]   # [filename] of currently checked row
_all_files = [None]  # [list] current unfiltered file list
_main_out = widgets.Output()
display(_main_out)


# ── action buttons ───────────────────────────────────────────────────────
def _make_action_bar():
    _new_btn    = widgets.Button(description='+ New Plan',  button_style='primary',
                                  layout=widgets.Layout(width='110px', height='32px'))
    _open_btn   = widgets.Button(description='Edit',        button_style='',
                                  layout=widgets.Layout(width='80px',  height='32px'), disabled=True)
    _pivot_btn  = widgets.Button(description='Pivot',       button_style='',
                                  layout=widgets.Layout(width='80px',  height='32px'), disabled=True)
    _delete_btn = widgets.Button(description='Delete',      button_style='danger',
                                  layout=widgets.Layout(width='80px',  height='32px'), disabled=True)
    _action_btns = [_open_btn, _pivot_btn, _delete_btn]

    def _set_actions(enabled):
        for b in _action_btns:
            b.disabled = not enabled

    return _new_btn, _open_btn, _pivot_btn, _delete_btn, _set_actions


# ── table renderer ───────────────────────────────────────────────────────
def _plans_folder():
    pid = _state.get('profile', {}).get('id', 'default')
    return f'plans/{pid}'


def _render_table(files=None):
    if files is None:
        files = drive_mgr.list_files(_plans_folder())
    _all_files[0] = files
    _selected[0] = None

    _new_btn, _open_btn, _pivot_btn, _delete_btn, _set_actions = _make_action_bar()
    _filter_input = widgets.Text(
        placeholder='Filter plans...',
        layout=widgets.Layout(width='260px'),
    )
    _table_out = widgets.Output()
    _status_out = widgets.HTML()
    _confirm_area = widgets.VBox([])
    _row_checks = {}  # filename -> Checkbox widget

    def _render_rows(filtered):
        _row_checks.clear()
        with _table_out:
            clear_output(wait=True)
            if not filtered:
                display(widgets.HTML(
                    '<p style="color:#94a3b8;font-size:13px;margin:12px 0">'
                    'No plans found. Click <b>+ New Plan</b> to get started.</p>'
                ))
                return
            # header
            header = widgets.HBox([
                widgets.HTML('<div style="width:28px"></div>'),
                widgets.HTML(_th('Name',   '240px')),
                widgets.HTML(_th('Goal',   '140px')),
                widgets.HTML(_th('Saved',  '100px')),
                widgets.HTML(_th('Cost',   '70px')),
                widgets.HTML(_th('Gaps',   '50px')),
            ])
            rows = [header]
            for f in reversed(filtered):
                m = f.get('meta', {})
                name  = m.get('name', f['name'])
                goal  = m.get('goal_type', '')
                saved = f['modified'][:10]
                cost  = f'${m["cost_usd"]:.4f}' if m.get('cost_usd') else ''
                _gap_count = len(m.get('gaps', []))
                _gap_cell = (f'<div style="width:50px;font-size:12px;color:#fbbf24;font-weight:600;padding:6px 0">'
                             f'{_gap_count}</div>') if _gap_count else '<div style="width:50px"></div>'
                chk = widgets.Checkbox(value=False, indent=False,
                                        layout=widgets.Layout(width='28px', height='20px'))
                _row_checks[f['name']] = chk

                def _on_check(change, fname=f['name']):
                    if change['new']:
                        # uncheck all others
                        for fn, cb in _row_checks.items():
                            if fn != fname and cb.value:
                                cb.value = False
                        _selected[0] = fname
                        _set_actions(True)
                    else:
                        if _selected[0] == fname:
                            _selected[0] = None
                            _set_actions(False)

                chk.observe(_on_check, names='value')
                rows.append(widgets.HBox([
                    chk,
                    widgets.HTML(_td(name,  '240px', bold=True)),
                    widgets.HTML(_td(goal,  '140px')),
                    widgets.HTML(_td(saved, '100px')),
                    widgets.HTML(_td(cost,  '70px')),
                    widgets.HTML(_gap_cell),
                ]))
            display(widgets.VBox(rows))

    def _on_filter(change):
        q = change['new'].lower()
        filtered = [f for f in _all_files[0]
                    if q in f.get('meta', {}).get('name', f['name']).lower()
                    or q in f.get('meta', {}).get('goal_type', '').lower()]
        _selected[0] = None
        _set_actions(False)
        _render_rows(filtered)

    _filter_input.observe(_on_filter, names='value')

    # ── action handlers ──
    def _on_new(btn):
        _saved = drive_mgr.list_files('profiles')
        _saved_meta = next((f['meta'] for f in _saved if f['name'] == 'default.json' and f.get('meta')), None)
        if _saved_meta:
            _state['profile'] = _saved_meta
        _state['goal'] = None
        _state['goal_info'] = {}
        _state['topics'] = []
        _state['current_topic'] = None
        _state['history'] = {}
        _state['locked'] = {}
        _state.pop('last_plan', None)
        with _main_out:
            clear_output(wait=True)
            _render_goal_selector(on_done=_render_table, seed_from_profile=True)

    def _on_open(btn):
        fname = _selected[0]
        if not fname:
            return
        content, meta = drive_mgr.load(_plans_folder(), fname)
        if not content:
            _status_out.value = ui.error_html(f'Could not load plan: {fname}')
            return
        with _main_out:
            clear_output(wait=True)
            _render_detail(fname, content, meta)

    def _on_pivot(btn):
        fname = _selected[0]
        if not fname:
            return
        _, meta = drive_mgr.load(_plans_folder(), fname)
        _lt = meta.get('locked_topics', {}) if meta else {}
        goal_type = meta.get('goal_type', '') if meta else ''
        flat = _lt.get(goal_type, _lt) if isinstance(next(iter(_lt.values()), None), dict) else _lt
        # seed pivot answers for opener injection -- history stays empty so opener fires
        _state['_pivot_answers'] = {tid: ans for tid, ans in flat.items() if ans}
        _state['history'] = {}
        _state['locked'] = {}
        _state['goal'] = None
        _state['_pivot_source'] = dict(flat)  # store answers for confirmation display
        with _main_out:
            clear_output(wait=True)
            _render_goal_selector(on_done=_render_table, pivot=True)

    def _on_delete(btn):
        fname = _selected[0]
        if not fname:
            return
        # confirm widget
        _conf_btn = widgets.Button(description='Confirm Delete', button_style='danger',
                                    layout=widgets.Layout(width='140px', height='32px'))
        _cancel_btn = widgets.Button(description='Cancel', button_style='',
                                      layout=widgets.Layout(width='80px', height='32px'))
        def _confirm(b):
            drive_mgr.delete(_plans_folder(), fname)
            _confirm_area.children = []
            _status_out.value = ui.warning_html(f'Deleted: {fname}')
            _render_table()
        def _cancel(b):
            _confirm_area.children = []
            _status_out.value = ''
        _conf_btn.on_click(_confirm)
        _cancel_btn.on_click(_cancel)
        _status_out.value = ui.error_html(f'Delete {fname}?')
        _confirm_area.children = [widgets.HBox([_conf_btn, _cancel_btn], layout=widgets.Layout(gap='8px'))]

    _new_btn.on_click(_on_new)
    _open_btn.on_click(_on_open)
    _pivot_btn.on_click(_on_pivot)
    _delete_btn.on_click(_on_delete)

    with _main_out:
        clear_output(wait=True)
        _prof_name = _state.get('profile', {}).get('name', 'Default')
        display(ui.card_widget([
            widgets.HTML(f'<p style="color:#475569;font-size:11px;margin:0 0 8px">Profile: <b style="color:#94a3b8">{_prof_name}</b></p>'),
            widgets.HBox([
                _filter_input,
                widgets.HBox([_new_btn, _open_btn, _pivot_btn, _delete_btn],
                              layout=widgets.Layout(gap='6px', margin='0 0 0 12px')),
            ], layout=widgets.Layout(align_items='center', margin='0 0 10px')),
            _table_out,
            _status_out,
            _confirm_area,
        ], title='My plans'))
    _render_rows(files)


# ── table cell helpers ────────────────────────────────────────────────────
def _th(label, width):
    return (f'<div style="width:{width};font-size:11px;font-weight:600;'
            f'color:#64748b;text-transform:uppercase;padding:4px 0">{label}</div>')

def _td(val, width, bold=False):
    weight = 'font-weight:500;' if bold else ''
    return (f'<div style="width:{width};font-size:13px;{weight}'
            f'color:#e2e8f0;padding:6px 0;white-space:nowrap;'
            f'overflow:hidden;text-overflow:ellipsis">{val}</div>')

# ── detail view ──────────────────────────────────────────────────────────
def _render_detail(fname, content, meta):
    _content = [content]  # mutable so refine apply can update it
    _back_btn = widgets.Button(description='< Back to Plans', button_style='',
                                layout=widgets.Layout(width='140px', height='32px'))
    _back_btn.on_click(lambda b: _render_table())

    _plan_name = [meta.get('name', fname) if meta else fname]
    plan_name = _plan_name[0]
    _file_path = str(drive_mgr.root / _plans_folder() / fname)

    # ── header name label ──
    _name_label = widgets.HTML(f'<span style="color:#f1f5f9;font-size:14px;margin-left:12px">{plan_name}</span>')

    # ── View tab ──
    _view_html = widgets.HTML(
        f'<div style="background:#1e293b;border:1px solid #475569;border-radius:8px;'
        f'padding:20px;max-height:460px;overflow-y:auto;font-family:Inter,system-ui,sans-serif">'
        f'{_md_to_html(_content[0])}</div>'
    )
    _prov_rows = []
    if meta:
        for k, v in [
            ('Goal',    meta.get('goal_type', '')),
            ('Target',  meta.get('target', '') or 'general'),
            ('Saved',   meta.get('created_at', '')),
            ('Cost',    f'${meta["cost_usd"]:.4f}' if meta.get('cost_usd') else ''),
            ('Model',   meta.get('model', '')),
        ]:
            if v:
                _prov_rows.append(f'<tr><td style="color:#64748b;font-size:12px;padding:3px 12px 3px 0;white-space:nowrap">{k}</td>'
                                  f'<td style="color:#e2e8f0;font-size:12px;padding:3px 0">{v}</td></tr>')
        locked = meta.get('locked_topics', {})
        goal_type_key = meta.get('goal_type', '')
        flat_locked = locked.get(goal_type_key, locked) if isinstance(next(iter(locked.values()), None), dict) else locked
        for tid, ans in flat_locked.items():
            label = tid.replace('_', ' ').title()
            _prov_rows.append(f'<tr><td style="color:#64748b;font-size:12px;padding:3px 12px 3px 0;white-space:nowrap;vertical-align:top">{label}</td>'
                              f'<td style="color:#cbd5e1;font-size:12px;padding:3px 0;white-space:pre-wrap;word-break:break-word">{str(ans)}</td></tr>')
    _prov_html = widgets.HTML(
        f'<div style="margin-top:12px"><p style="color:#64748b;font-size:11px;font-weight:600;'
        f'text-transform:uppercase;margin:0 0 6px">Provenance</p>'
        f'<table style="border-collapse:collapse">{chr(10).join(_prov_rows)}</table></div>'
    )
    _rename_input = widgets.Text(value=plan_name, layout=widgets.Layout(width='300px'))
    _rename_save_btn = widgets.Button(description='Save name', button_style='success',
                                       layout=widgets.Layout(width='100px', height='32px'))
    _rename_out = widgets.HTML()
    def _on_rename_save(b):
        new_name = _rename_input.value.strip()
        if not new_name:
            return
        updated_meta = dict(meta) if meta else {}
        updated_meta['name'] = new_name
        drive_mgr.save(_plans_folder(), fname, content, meta=updated_meta)
        _plan_name[0] = new_name
        _name_label.value = f'<span style="color:#f1f5f9;font-size:14px;margin-left:12px">{new_name}</span>'
        _rename_out.value = ui.success_html('Renamed.')
    _rename_save_btn.on_click(_on_rename_save)
    _edit_disc_btn = widgets.Button(description='Edit in Discovery', button_style='primary',
                                     layout=widgets.Layout(width='150px', height='32px', margin='12px 0 0'))
    _edit_out = widgets.HTML()
    def _on_edit_disc(b):
        _edit_out.value = ui.warning_html('Loading...')
        if not meta:
            _edit_out.value = ui.error_html('No metadata found.')
            return
        goal_type = meta.get('goal_type')
        _state['goal'] = goal_type
        _lt = meta.get('locked_topics', {})
        _state['locked'] = dict(_lt.get(goal_type, _lt) if isinstance(next(iter(_lt.values()), None), dict) else _lt)
        _state['history'] = {}
        _goals_result = client.call('GET', '/goals')
        if _goals_result['ok']:
            _goal_def = next((g for g in _goals_result['data']['goals'] if g['id'] == goal_type), None)
            if _goal_def:
                _state['goal_info'] = _goal_def
                _state['topics'] = [{'id': t['id'], 'name': t['name'], 'opener': t.get('opener', '')}
                                    for t in _goal_def.get('required_topics', [])]
                unlocked = [t for t in _state['topics'] if t['id'] not in _state['locked']]
                _state['current_topic'] = unlocked[0]['id'] if unlocked else (_state['topics'][0]['id'] if _state['topics'] else None)
        _edit_out.value = (
            ui.success_html('Ready.')
            + ui.instruction_html('Scroll down to the Discovery chat cell and click Run.')
        )
    _edit_disc_btn.on_click(_on_edit_disc)
    _drive_rel = _file_path.replace(str(drive_mgr.root), '').lstrip('/')
    _file_link = widgets.HTML(
        '<p style="color:#475569;font-size:11px;margin:10px 0 0">'
        'Open and edit directly in '
        '<a href="https://drive.google.com/drive/home" target="_blank" '
        'style="color:#3b82f6">Google Drive</a> &mdash; '
        f'My Drive / pysolvr / {_drive_rel}</p>'
    )
    _view_tab = widgets.VBox([_view_html, _prov_html, _file_link,
        widgets.HBox([_rename_input, _rename_save_btn],
                      layout=widgets.Layout(align_items='center', gap='6px', margin='12px 0 4px')),
        _rename_out])

    # ── Export tab ──
    _doc_dd = widgets.Dropdown(
        options=['Executive Summary', 'Pitch Deck', 'One Pager', 'Financial Summary', 'Full Report'],
        description='Document:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='280px'),
    )
    _filetype_dd = widgets.Dropdown(
        options=['Markdown', 'Plain text', 'PDF'],
        description='File type:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='200px'),
    )
    _exp_btn = widgets.Button(description='Preview', button_style='primary',
                               layout=widgets.Layout(width='90px', height='34px'))
    _exp_status = widgets.HTML()
    _exp_result = widgets.VBox([])
    _exp_async_out = widgets.Output()
    def _on_export(b):
        _exp_btn.disabled = True
        _exp_status.value = ui.warning_html('Generating...')
        _exp_result.children = []
        try:
            _ext = {'Markdown': '.md', 'Plain text': '.txt', 'PDF': '.pdf'}.get(_filetype_dd.value, '.md')
            _doc_slug = _doc_dd.value.lower().replace(' ', '_')
            result = client.run_async('/export', {'content': content,
                                                  'format': _doc_slug,
                                                  'file_type': _filetype_dd.value.lower().replace(' ', '_')},
                                      output=_exp_async_out)
            _exp_status.value = ''
            if result['ok']:
                data = result['data']
                _default_exp_fname = f"{_doc_slug}_{fname.rsplit('.', 1)[0]}{_ext}"
                _fname_input = widgets.Text(value=_default_exp_fname,
                                            layout=widgets.Layout(width='380px'),
                                            style={'description_width': '0px'})
                _save_exp_btn = widgets.Button(description='Save to Drive', button_style='success',
                                               layout=widgets.Layout(width='120px', height='32px'))
                _exp_save_status = widgets.HTML()
                def _do_save(_):
                    _save_exp_btn.disabled = True
                    out_fname = _fname_input.value.strip() or _default_exp_fname
                    exp_path = drive_mgr.save('exports', out_fname, data['content'],
                                    meta={'document_type': _doc_slug, 'file_type': _filetype_dd.value,
                                          'source_plan': fname, 'source_name': _plan_name[0]})
                    _exp_rel = exp_path.replace(str(drive_mgr.root), '').lstrip('/')
                    _exp_save_status.value = (
                        ui.success_html(f'Saved: {out_fname}')
                        + ui.instruction_html(f'My Drive / pysolvr / {_exp_rel}')
                    )
                _save_exp_btn.on_click(_do_save)
                _exp_result.children = [
                    widgets.HTML('<p style="color:#94a3b8;font-size:11px;margin:0 0 4px">Preview (edit directly in Drive after saving):</p>'),
                    widgets.HTML(
                        f'<pre style="white-space:pre-wrap;color:#f1f5f9;font-size:12px;'
                        f'background:#1e293b;border:1px solid #334155;border-radius:6px;'
                        f'padding:12px;max-height:300px;overflow-y:auto;margin:0 0 10px">{data["content"]}</pre>'
                    ),
                    widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:0 0 4px">Filename:</p>'),
                    widgets.HBox([_fname_input, _save_exp_btn],
                                 layout=widgets.Layout(align_items='center', gap='8px')),
                    _exp_save_status,
                ]
            else:
                _err = result.get('error', 'Export failed')
                _lk = result.get('limit_key')
                if _lk:
                    _exp_result.children = [widgets.HTML(
                        '<div style="background:#1e1a2e;border:1px solid #7c3aed;border-radius:8px;padding:12px 16px;margin:8px 0">'
                        f'<p style="color:#f87171;font-size:13px;margin:0 0 6px">{_err}</p>'
                        '<a href="https://planforge.ai/#pricing" target="_blank" '
                        'style="display:inline-block;background:#7c3aed;color:#fff;font-size:12px;'
                        'padding:6px 14px;border-radius:6px;text-decoration:none;font-weight:600">Upgrade &rarr;</a>'
                        '</div>'
                    )]
                else:
                    _exp_result.children = [widgets.HTML(ui.error_html(_err))]
        except Exception as _e:
            _exp_status.value = ''
            _exp_result.children = [widgets.HTML(ui.error_html(f'Export failed: {_e}'))]
        finally:
            _exp_btn.disabled = False
    _exp_btn.on_click(_on_export)
    _export_tab = widgets.VBox([
        widgets.HBox([_doc_dd, _filetype_dd, _exp_btn],
                      layout=widgets.Layout(align_items='center', gap='8px', margin='8px 0')),
        _exp_status,
        _exp_result,
    ])

    # ── Refine tab ──
    _locked_topics_raw = meta.get('locked_topics', {}) if meta else {}
    _goal_type_key = meta.get('goal_type', '') if meta else ''
    _locked_topics = (_locked_topics_raw.get(_goal_type_key, _locked_topics_raw) if isinstance(next(iter(_locked_topics_raw.values()), None), dict) else _locked_topics_raw) if _locked_topics_raw else {}
    _gaps = meta.get('gaps', []) if meta else []
    _section_titles = meta.get('section_titles', {}) if meta else {}
    # pills keyed by section title (what the LLM actually wrote) not topic id
    _refine_sections = list(_section_titles.values()) if _section_titles else [tid.replace('_', ' ').title() for tid in _locked_topics]
    _selected_topic = [None]
    _fb_input  = widgets.Textarea(placeholder='What to improve? (optional)',
                                   layout=widgets.Layout(width='100%', max_width='560px', height='70px'))
    _ref_btn     = widgets.Button(description='Refine', button_style='primary',
                                   layout=widgets.Layout(width='90px', height='34px'), disabled=True)
    _ref_status  = widgets.HTML()
    _ref_result  = widgets.VBox([])
    _ref_preview = widgets.HTML()
    _ref_async_out = widgets.Output()
    def _make_section_pill(title):
        b = widgets.Button(description=title,
                           layout=widgets.Layout(height='28px', margin='2px', padding='0 10px'))
        def _on_pill(_, t=title):
            _selected_topic[0] = t
            _ref_btn.disabled = False
            for pb in _ref_pills:
                pb.button_style = 'primary' if pb._section_title == t else ''
            import re as _re2
            _parts = _re2.split(r'(^#{1,3}\s+.+$)', content, flags=_re2.MULTILINE)
            _secs2, _ch2, _cb2 = [], None, ''
            for _p in _parts:
                if _re2.match(r'^#{1,3}\s+', _p):
                    if _ch2: _secs2.append((_ch2, _cb2.strip()))
                    _ch2 = _p.strip(); _cb2 = ''
                else: _cb2 += _p
            if _ch2: _secs2.append((_ch2, _cb2.strip()))
            _t_lower = t.lower()
            _match = next((body for h, body in _secs2 if _t_lower in h.lower()), None)
            preview_text = _match[:400] + ('...' if _match and len(_match) > 400 else '') if _match else '(section not found in plan)'
            _ref_preview.value = (
                f'<div style="background:#1e293b;border:1px solid #334155;border-radius:6px;'
                f'padding:10px 14px;margin:8px 0;font-size:12px;color:#cbd5e1;'
                f'white-space:pre-wrap;max-height:120px;overflow-y:auto">{preview_text}</div>'
            )
        b._section_title = title
        b.on_click(_on_pill)
        return b
    _ref_pills = [_make_section_pill(t) for t in _refine_sections]
    # ── gap panel ──
    def _select_pill_by_section(section_id):
        # section_id from gaps matches section_titles keys; find the title value
        _target_title = _section_titles.get(section_id, section_id.replace('_', ' ').title())
        for pb in _ref_pills:
            if pb._section_title == _target_title:
                pb.fire_event('click', {})
                break
    _gap_widgets = []
    for _g in _gaps:
        _g_btn = widgets.Button(description='Refine this section',
                                layout=widgets.Layout(width='150px', height='28px'))
        def _on_gap_btn(_, sid=_g['section_id']):
            _select_pill_by_section(sid)
        _g_btn.on_click(_on_gap_btn)
        _gap_widgets.append(widgets.VBox([
            widgets.HTML(
                '<div style="background:#1e293b;border:1px solid #f59e0b;border-radius:6px;padding:10px 14px;margin:3px 0">'
                f'<p style="color:#f1f5f9;font-size:13px;font-weight:600;margin:0 0 1px">{_g["label"]}</p>'
                f'<p style="color:#64748b;font-size:11px;text-transform:uppercase;margin:0 0 5px">{_g["section_title"]}</p>'
                f'<p style="color:#94a3b8;font-size:12px;margin:0 0 5px">{_g["why_it_matters"]}</p>'
                f'<p style="color:#cbd5e1;font-size:12px;margin:0"><b style="color:#e2e8f0">How to address: </b>{str(_g["how_to_satisfy"]).splitlines()[0]}</p>'
                '</div>'
            ),
            _g_btn,
        ]))
    _gap_panel = widgets.VBox([
        widgets.HTML(
            '<div style="background:#1e293b;border:1px solid #f59e0b;border-radius:6px;padding:8px 14px;margin:0 0 8px">'
            f'<p style="color:#fbbf24;font-size:13px;font-weight:600;margin:0 0 2px">\u26a0\ufe0f {len(_gaps)} gap(s) identified</p>'
            '<p style="color:#94a3b8;font-size:12px;margin:0">These items were missing when the plan was generated. '
            'Click "Refine this section" on a gap to address it.</p></div>'
        ),
        *_gap_widgets,
        widgets.HTML('<div style="height:8px"></div>'),
    ]) if _gaps else widgets.HTML()
    def _patch_section(plan_md, section_name, new_body):
        import re as _re3
        _lines = plan_md.split('\n')
        _sec_lower = section_name.lower()
        _start = None; _level = None
        for i, ln in enumerate(_lines):
            m = _re3.match(r'^(#{1,3})\s+(.+)$', ln)
            if m and _sec_lower in m.group(2).lower():
                _start = i; _level = len(m.group(1)); break
        if _start is None: return plan_md
        _end = len(_lines)
        for i in range(_start + 1, len(_lines)):
            m = _re3.match(r'^(#{1,3})\s+', _lines[i])
            if m and len(m.group(1)) <= _level: _end = i; break
        return '\n'.join(_lines[:_start + 1] + [''] + new_body.strip().split('\n') + [''] + _lines[_end:])
    def _on_refine(b):
        if not _selected_topic[0]:
            _ref_status.value = ui.error_html('Select a topic first.')
            return
        _ref_btn.disabled = True
        _ref_status.value = ui.warning_html('Refining...')
        _ref_result.children = []
        body = {'content': _content[0], 'section': _selected_topic[0]}
        if _fb_input.value.strip():
            body['feedback'] = _fb_input.value.strip()
        try:
            result = client.run_async('/refine', body, output=_ref_async_out)
        except Exception as _e:
            _ref_status.value = ''
            _ref_result.children = [widgets.HTML(ui.error_html(f'Refinement failed: {_e}'))]
            _ref_btn.disabled = False
            return
        _ref_status.value = ''
        _ref_btn.disabled = False
        if result['ok']:
            data = result['data']
            _apply_btn = widgets.Button(description='Apply & Save', button_style='success',
                                        layout=widgets.Layout(width='120px', height='32px'))
            _apply_status = widgets.HTML()
            def _on_apply(_):
                _apply_btn.disabled = True
                _patched = _patch_section(_content[0], data['section'], data['content'])
                saved_path = drive_mgr.save(_plans_folder(), fname, _patched, meta=meta)
                _content[0] = _patched
                _view_html.value = (
                    '<div style="background:#1e293b;border:1px solid #475569;border-radius:8px;'
                    'padding:20px;max-height:460px;overflow-y:auto;font-family:Inter,system-ui,sans-serif">'
                    + _md_to_html(_patched) + '</div>'
                )
                _rel = saved_path.replace(str(drive_mgr.root), '').lstrip('/')
                _apply_status.value = (
                    ui.success_html('Applied and saved.')
                    + ui.instruction_html('Tweak it further in '
                                         f'<a href="https://drive.google.com/drive/home" target="_blank" style="color:#3b82f6">Google Drive</a>'
                                         f' -- My Drive / pysolvr / {_rel}')
                )
            _apply_btn.on_click(_on_apply)
            _cost = f'  &mdash; {data["tokens_used"]} tokens / ${data["cost_usd"]:.4f}' if data.get('cost_usd') else ''
            _ref_result.children = [
                widgets.HTML(f'<p style="color:#94a3b8;font-size:11px;margin:0 0 4px">Refined: <b style="color:#f1f5f9">{data["section"]}</b>{_cost}</p>'),
                widgets.HTML(
                    f'<pre style="white-space:pre-wrap;color:#f1f5f9;font-size:12px;background:#1e293b;'
                    f'border:1px solid #334155;border-radius:6px;padding:12px;max-height:260px;overflow-y:auto;margin:0 0 8px">'
                    f'{data["content"]}</pre>'
                ),
                widgets.HBox([_apply_btn], layout=widgets.Layout(margin='0 0 4px')),
                _apply_status,
            ]
        else:
            _ref_result.children = [widgets.HTML(ui.error_html(result.get('error', 'Refinement failed')))]
    _ref_btn.on_click(_on_refine)
    _refine_tab = widgets.VBox([
        _gap_panel,
        widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:4px 0 6px">Select a topic to refine a specific section in-place:</p>'),
        widgets.HBox(_ref_pills, layout=widgets.Layout(flex_flow='row wrap', margin='0 0 4px')),
        _ref_preview,
        widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:8px 0 4px">Feedback (optional)</p>'),
        _fb_input,
        widgets.HBox([_ref_btn], layout=widgets.Layout(margin='8px 0 0')),
        _ref_status,
        _ref_result,
        widgets.HTML('<hr style="border-color:#334155;margin:16px 0 10px">'),
        widgets.HTML('<p style="color:#94a3b8;font-size:12px;margin:0 0 6px">Or re-run Discovery to change your answers and regenerate the whole plan:</p>'),
        _edit_disc_btn, _edit_out,
    ])

    # -- Tabs --
    _tab_style = widgets.HTML(
        '<style>'
        '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab,'
        '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current,'
        '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab:hover {'
        'background:transparent !important;color:#94a3b8;'
        'border-width:0 0 2px 0 !important;border-style:solid !important;'
        'border-color:transparent !important;box-shadow:none !important;'
        'font-size:13px;font-family:Inter,system-ui,sans-serif;padding:8px 16px;}'
        '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current {'
        'color:#f1f5f9 !important;border-bottom-color:#3b82f6 !important;}'
        '.jupyter-widgets.widget-tab > .p-TabBar {border-bottom:1px solid #475569;}'
        '.jupyter-widgets.widget-tab > .widget-tab-contents {'
        'border:none;padding:12px 0 0;background:transparent;}'
        '</style>'
    )
    _tabs = widgets.Tab(children=[_view_tab, _refine_tab, _export_tab])
    for i, title in enumerate(['View', 'Refine', 'Export']):
        _tabs.set_title(i, title)

    with _main_out:
        clear_output(wait=True)
        display(ui.card_widget([
            widgets.HBox([_back_btn, _name_label],
                         layout=widgets.Layout(align_items='center', margin='0 0 12px')),
            _tab_style,
            _tabs,
        ], title='My plans'))
# ── goal selector (New Plan + Pivot) ─────────────────────────────────────
def _render_goal_selector(on_done, pivot=False, seed_from_profile=False):
    _gs_out = widgets.Output()
    _gs_card = ui.card_widget([_gs_out], title='What do you want to achieve?')
    _CATEGORY_LABELS = {
        'Grants': 'I need a grant',
        'Loans': 'I need a loan',
        'Investment': 'I want to raise investment',
        'Visas': 'I need a visa for my business',
        'Accelerators': "I'm applying to an accelerator",
        'Validation': 'I want to test my idea',
    }
    _CAT_ORDER = ['Grants', 'Loans', 'Investment', 'Visas', 'Accelerators', 'Validation']

    def _pill(label, on_click):
        b = widgets.Button(description=label,
                           layout=widgets.Layout(margin='4px', height='36px', width='auto'))
        b.style.button_color = '#1e293b'
        b.style.text_color = 'white'
        b.on_click(on_click)
        return b

    result = client.call('GET', '/goals')
    if not result['ok']:
        with _gs_out:
            ui.error('Could not load goals', result.get('error', ''))
        display(_gs_out)
        return

    goals = result['data']['goals']
    categories = {}
    for g in goals:
        categories.setdefault(g.get('category', 'Other'), []).append(g)
    sorted_cats = [c for c in _CAT_ORDER if c in categories]
    sorted_cats += [c for c in categories if c not in _CAT_ORDER]

    title = 'Pivot: choose new goal type' if pivot else 'What do you want to achieve?'

    def _cancel_btn():
        b = widgets.Button(description='Cancel', button_style='',
                           layout=widgets.Layout(width='80px', height='32px', margin='8px 0 0'))
        b.on_click(lambda _: on_done())
        return b

    def _show_cats():
        with _gs_out:
            clear_output(wait=True)
            btns = [_pill(_CATEGORY_LABELS.get(c, c), lambda _, cat=c: _show_goals(cat))
                    for c in sorted_cats]
            display(widgets.VBox([
                widgets.HTML(f'<p style="color:#f1f5f9;font-size:14px;margin:0 0 10px">{title}</p>'),
                widgets.HBox(btns, layout=widgets.Layout(flex_flow='row wrap')),
                _cancel_btn(),
            ]))

    def _show_goals(category):
        cat_goals = categories[category]
        if len(cat_goals) == 1 and not cat_goals[0].get('targets'):
            _confirm(cat_goals[0])
            return
        with _gs_out:
            clear_output(wait=True)
            back = _pill('< Back', lambda _: _show_cats())
            btns = [back] + [_pill(g['name'], lambda _, goal=g: _confirm(goal))
                             for g in cat_goals]
            display(widgets.VBox([
                widgets.HTML(f'<p style="color:#f1f5f9;font-size:14px;margin:0 0 10px">{_CATEGORY_LABELS.get(category, category)}</p>'),
                widgets.HBox(btns, layout=widgets.Layout(flex_flow='row wrap')),
                _cancel_btn(),
            ]))

    def _confirm(goal):
        _state['goal'] = goal['id']
        _state['goal_info'] = goal
        _state['current_topic'] = None
        _state['history'] = {}
        _state['topics'] = [{'id': t['id'], 'name': t['name'], 'opener': t.get('opener', '')}
                             for t in goal.get('required_topics', [])]
        new_topic_ids = {t['id'] for t in _state['topics']}
        if seed_from_profile:
            _lt = _state.get('profile', {}).get('locked_topics', {})
            if _lt and not isinstance(next(iter(_lt.values()), None), dict):
                all_locked = _lt
            else:
                all_locked = {}
                for bucket in _lt.values():
                    if isinstance(bucket, dict):
                        all_locked.update(bucket)
            _state['locked'] = {k: v for k, v in all_locked.items() if k in new_topic_ids}
        else:
            existing_locked = _state.get('locked', {})
            if _state.get('_pivot_source'):
                # pivot: keep all answers as context, don't filter by new topic IDs
                _state['locked'] = dict(existing_locked)
            else:
                _state['locked'] = {k: v for k, v in existing_locked.items() if k in new_topic_ids}
        # set current_topic to first unlocked
        for t in _state['topics']:
            if t['id'] not in _state['locked']:
                _state['current_topic'] = t['id']
                break
        else:
            _state['current_topic'] = _state['topics'][0]['id'] if _state['topics'] else None
        covered = len(_state['locked'])
        _pivot_answers = _state.get('_pivot_answers', {})
        pivot_headstart = [t for t in _state['topics'] if t['id'] in _pivot_answers] if _pivot_answers else []
        total   = len(_state['topics'])
        gap_note = ''
        if covered < total:
            missing = [t['name'] for t in _state['topics'] if t['id'] not in _state['locked']]
            if pivot_headstart:
                gap_note = (f'<p style="color:#fbbf24;font-size:12px;margin:8px 0 0">'
                            f'Pivot headstart on {len(pivot_headstart)}/{total} topics: {", ".join(t["name"] for t in pivot_headstart)}</p>'
                            f'<p style="color:#94a3b8;font-size:12px;margin:4px 0 0">'
                            f'{total - len(pivot_headstart)} topic(s) are new: {", ".join(t["name"] for t in _state["topics"] if t["id"] not in _pivot_answers)}</p>')
            else:
                gap_note = (f'<p style="color:#fbbf24;font-size:12px;margin:8px 0 0">'
                            f'{len(missing)} topic(s) need covering: {", ".join(missing)}</p>')
        _pivot_block = ''
        _pivot_answers = _state.pop('_pivot_source', None)
        if _pivot_answers:
            _rows = ''.join(f'<tr><td style="color:#94a3b8;font-size:12px;padding:3px 10px 3px 0;white-space:nowrap;vertical-align:top">{tid.replace("_"," ").title()}</td><td style="color:#e2e8f0;font-size:12px;padding:3px 0;white-space:pre-wrap">{str(ans)[:300]}</td></tr>' for tid, ans in _pivot_answers.items() if ans)
            _pivot_block = ('<div style="background:#1e293b;border:1px solid #334155;border-radius:8px;padding:12px 16px;margin:10px 0 0">'
                           '<p style="color:#94a3b8;font-size:12px;font-weight:600;margin:0 0 6px">\ud83d\udccc From your previous plan &mdash; copy and paste anything useful into the chat, or tweak it for this new goal:</p>'
                           f'<table style="border-collapse:collapse;width:100%">{_rows}</table></div>')
        with _gs_out:
            clear_output(wait=True)
            display(widgets.HTML(
                f'<p style="color:#10b981;font-size:13px;margin:0 0 4px">'
                + (f'Goal set: <b>{goal["name"]}</b> &mdash; pivot headstart on {len(pivot_headstart)}/{total} topics.</p>' if pivot_headstart else f'Goal set: <b>{goal["name"]}</b> &mdash; {covered}/{total} topics already covered.</p>')
                + f'{gap_note}'
                + f'{_pivot_block}'
                + f'<p style="color:#94a3b8;font-size:12px;margin:6px 0 0">'
                + f'Scroll down to the Discovery chat cell and click Run to continue.</p>'
            ))

    display(_gs_card)
    _show_cats()


# ── entry point ───────────────────────────────────────────────────────────
_render_table()


In [ ]:
# @title Discovery chat

from IPython.display import HTML

if not _state.get('goal') or not _state.get('topics'):
    ui.error('No goal selected', suggestion='Run the goal selection cell above first.')
    assert False, 'Goal not set'

if not isinstance(_state.get('history'), dict):
    _state['history'] = {}

# --- Widgets ---
_counter_out = widgets.Output()
_chat_out = widgets.Output(layout={'border': '1px solid #475569', 'border_radius': '8px',
                                    'padding': '16px', 'max_height': '400px',
                                    'overflow_y': 'auto', 'background': '#1e293b'})
_input_area = widgets.Textarea(placeholder='Type your message or paste text (CV, financials, notes)...',
                               layout=widgets.Layout(width='100%', height='80px'))
_send_btn = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='80px', height='34px'))
_clear_btn = widgets.Button(description='Clear', button_style='warning',
                             layout=widgets.Layout(width='80px', height='34px'))
_status_out = widgets.HTML()
_completion_out = widgets.HTML()
_lock_pending = [False]
_suggested_lock = [None]

# Topic pill buttons
_pill_btns = []
for _t in _state['topics']:
    _b = widgets.Button(description=_t['name'],
                        layout=widgets.Layout(height='26px', margin='2px', padding='0 8px',
                                             border_radius='12px'))
    _b._topic_id = _t['id']
    _pill_btns.append(_b)


def _all_locked():
    return all(t['id'] in _state['locked'] for t in _state['topics'])


def _get_history():
    topic = _state['current_topic'] or 'default'
    if topic not in _state['history']:
        _state['history'][topic] = []
    return _state['history'][topic]


def _show_completion():
    if _all_locked():
        _completion_out.value = (
            ui.success_html('All topics locked.')
            + ui.instruction_html(['Scroll down to Generate my plan and click Run.',
                                   'Or click any topic above to refine your answers.'])
        )
    else:
        _completion_out.value = ''


def _render_progress():
    with _counter_out:
        clear_output(wait=True)
        done = sum(1 for t in _state['topics'] if t['id'] in _state['locked'])
        total = len(_state['topics'])
        pct = int(done / total * 100) if total else 0
        color = '#10b981' if pct < 100 else '#6366f1'
        display(HTML(f'<div style="font-family:Inter,system-ui,sans-serif;color:#f1f5f9;margin-bottom:8px">'
                     f'<p style="font-size:13px;margin:0 0 6px;color:#94a3b8">{done}/{total} topics locked</p>'
                     f'<div style="height:8px;border-radius:4px;background:#1e293b;overflow:hidden">'
                     f'<div style="height:100%;width:{pct}%;background:{color};border-radius:4px;transition:width 0.3s"></div>'
                     f'</div></div>'))
    for b in _pill_btns:
        tid = b._topic_id
        name = next(t['name'] for t in _state['topics'] if t['id'] == tid)
        if tid in _state['locked']:
            b.button_style = 'success'
            b.description = '\u2713 ' + name
        elif tid == _state['current_topic']:
            b.button_style = 'primary'
            b.description = '\u25cf ' + name
        else:
            b.button_style = ''
            b.description = name
    _show_completion()


def _render_chat(show_lock=False):
    with _chat_out:
        clear_output(wait=True)
        topic = _state['current_topic'] or 'default'
        if topic in _state['locked']:
            locked_val = _state['locked'][topic]
            display(HTML(f'<div style="background:#064e3b;border:1px solid #10b981;border-radius:6px;'
                         f'padding:8px 12px;margin:0 0 10px;font-size:12px;color:#6ee7b7">'
                         f'<b>\u2713 Locked:</b> {locked_val}</div>'))
        msgs = _get_history()
        if not msgs:
            display(HTML(''))
            return
        html = ''
        for msg in msgs:
            if msg['role'] == 'user':
                text = msg['content'][:500] + ('...' if len(msg['content']) > 500 else '')
                html += (f'<div style="text-align:right;margin:6px 0">'
                         f'<span style="background:#1e40af;color:#f1f5f9;padding:8px 12px;'
                         f'border-radius:14px 14px 4px 14px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{text}</span></div>')
            else:
                html += (f'<div style="text-align:left;margin:6px 0">'
                         f'<span style="background:#334155;color:#e2e8f0;padding:8px 12px;'
                         f'border-radius:14px 14px 14px 4px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{msg["content"]}</span></div>')
        display(HTML(html))
        if show_lock:
            display(HTML('<p style="margin:12px 0 4px;font-size:14px;color:#f1f5f9"><b>Ready to lock this topic in?</b></p>'))
            _inline_lock = widgets.Button(description='\u2713 Lock in', button_style='success',
                                          layout=widgets.Layout(width='120px', height='34px', margin='4px 0 0 0'))
            _inline_lock.on_click(_do_lock)
            display(_inline_lock)


_LOCK_PHRASES = {'lock', 'lock in', 'lock this', 'lock this in', 'yes', 'yep', 'correct', 'thats right', 'confirmed', 'confirm', 'done'}


def _switch_topic(btn):
    topic_id = btn._topic_id
    if topic_id == _state['current_topic'] and topic_id not in _state['locked']:
        return
    _state['current_topic'] = topic_id
    _lock_pending[0] = False
    _suggested_lock[0] = None
    # inject opener if this unlocked topic has never been visited
    if topic_id not in _state['locked'] and not _state['history'].get(topic_id):
        _t_info = next((t for t in _state['topics'] if t['id'] == topic_id), {})
        opener = _t_info.get('opener') or f'Tell me about your {_t_info.get("name", "").lower()}.'
        _pivot_ans = _state.get('_pivot_answers', {}).get(topic_id)
        if _pivot_ans:
            opener += f'\n\n---\n\ud83d\udd04 From your previous plan:\n{_pivot_ans}\n\nCopy and paste this into chat if it works for this new goal, or tweak it to suit.'
        _state['history'].setdefault(topic_id, []).append({'role': 'assistant', 'content': opener})
    _render_progress()
    _render_chat()
    _status_out.value = ''


for _b in _pill_btns:
    _b.on_click(_switch_topic)


def _on_clear(btn):
    topic = _state['current_topic'] or 'default'
    _state['history'][topic] = []
    _lock_pending[0] = False
    _render_chat()
    _status_out.value = ''


def _on_send(btn):
    message = _input_area.value.strip()
    if not message:
        return
    # lock intent: handle client-side without API round-trip
    if message.lower().rstrip('.!') in _LOCK_PHRASES and _state['current_topic']:
        _input_area.value = ''
        _do_lock()
        return
    # if current topic is locked and user is typing, unlock it first
    topic = _state['current_topic']
    if topic and topic in _state['locked']:
        prev_answer = _state['locked'].pop(topic)
        history = _state['history'].setdefault(topic, [])
        if not history:
            history.append({'role': 'user', 'content': prev_answer})
    msgs = _get_history()
    msgs.append({'role': 'user', 'content': message})
    _input_area.value = ''
    _lock_pending[0] = False
    _render_chat()
    _status_out.value = ui.warning_html('\u23f3 Thinking...')
    topic = _state['current_topic'] or 'business_overview'
    body = {'topic': topic, 'message': message, 'goal_type': _state['goal'],
            'facts': _state.get('profile', {}).get('facts', {}),
            'conversation_history': [{'role': m['role'], 'content': m['content'][:1000]}
                                     for m in msgs[-6:]],
            'locked_topics': _state['locked']}
    submit = client.call('POST', '/chat', body)
    if not submit['ok']:
        _status_out.value = ui.error_html(submit.get('error', 'Failed'), submit.get('actions'))
        return
    job_id = submit['data'].get('job_id')
    if not job_id:
        reply = submit['data'].get('response', '')
        msgs.append({'role': 'assistant', 'content': reply})
        _status_out.value = ''
        _render_chat()
        return
    for _ in range(60):
        _time.sleep(2)
        poll = client.call('GET', f'/jobs/{job_id}')
        if not poll['ok']:
            break
        if poll['data'].get('status') == 'complete':
            rd = poll['data'].get('result', {})
            msgs.append({'role': 'assistant', 'content': rd.get('response', '')})
            show_lock = rd.get('topic_status') in ('suggested_lock', 'locked')
            if show_lock:
                _suggested_lock[0] = rd.get('suggested_lock')
            if rd.get('topic_status') == 'locked':
                _do_lock()
            else:
                _lock_pending[0] = show_lock
                _status_out.value = ''
                _render_progress()
                _render_chat(show_lock=show_lock)
            return
        elif poll['data'].get('status') == 'failed':
            msgs.append({'role': 'assistant', 'content': f"Error: {poll['data'].get('error', 'Failed')}"})
            break
    _status_out.value = ''
    _render_progress()
    _render_chat()


def _save_profile():
    profile = _state.get('profile')
    if not profile:
        return
    goal_type = _state.get('goal', 'default')
    profile.setdefault('locked_topics', {})[goal_type] = dict(_state['locked'])
    drive_mgr.save('profiles', f"{profile['id']}.json", '', meta=profile)


def _do_lock(btn=None):
    topic = _state['current_topic']
    if not topic:
        return
    msgs = _state['history'].get(topic, [])
    summary = '\n'.join(m['content'] for m in msgs if m['role'] == 'user')
    _state['locked'][topic] = _suggested_lock[0] or summary or 'No details provided'
    _suggested_lock[0] = None
    _save_profile()
    _lock_pending[0] = False
    locked_name = next((t['name'] for t in _state['topics'] if t['id'] == topic), topic)
    next_topic = None
    for t in _state['topics']:
        if t['id'] not in _state['locked']:
            next_topic = t
            _state['current_topic'] = t['id']
            break
    _status_out.value = ''
    _render_progress()
    with _chat_out:
        clear_output(wait=True)
        if next_topic:
            opener = next_topic.get('opener') or f'Tell me about your {next_topic["name"].lower()}.'
            _state['history'].setdefault(next_topic['id'], []).insert(0, {'role': 'assistant', 'content': opener})
    _render_chat()


_entering = [False]


def _on_enter(change):
    if _entering[0]:
        return
    if change['new'].endswith('\n') and not change['old'].endswith('\n'):
        _entering[0] = True
        _input_area.value = change['new'].rstrip('\n')
        _entering[0] = False
        _on_send(None)
_input_area.observe(_on_enter, names='value')
_send_btn.on_click(_on_send)
_clear_btn.on_click(_on_clear)
_goal_label = _state.get('goal_info', {}).get('name', '')
_card_title = f'Discovery chat ({_goal_label})' if _goal_label else 'Discovery chat'
_prof_name = _state.get('profile', {}).get('name', 'Default')
_card = ui.card_widget([
    widgets.HTML(f'<p style="color:#475569;font-size:11px;margin:0 0 8px">Profile: <b style="color:#94a3b8">{_prof_name}</b></p>'),
    _counter_out,
    widgets.HBox(_pill_btns, layout=widgets.Layout(flex_flow='row wrap', margin='0 0 8px 0')),
    _chat_out,
    widgets.HBox([_input_area, widgets.VBox([_send_btn, _clear_btn])]),
    _status_out,
    _completion_out,
], title=_card_title)
display(_card)
if _state['current_topic'] and not _get_history():
    _t_info = next((t for t in _state['topics'] if t['id'] == _state['current_topic']), {})
    _opener = _t_info.get('opener') or f'Tell me about your {_t_info.get("name", "").lower()}.'
    _pivot_ans = _state.get('_pivot_answers', {}).get(_state['current_topic'])
    if _pivot_ans:
        _opener += f'\n\n---\n\ud83d\udd04 From your previous plan:\n{_pivot_ans}\n\nCopy and paste this into chat if it works for this new goal, or tweak it to suit.'
    _get_history().append({'role': 'assistant', 'content': _opener})
_render_progress()
_render_chat()


In [ ]:
# @title Generate my plan

import re as _re
from datetime import date as _date

def _md_to_html(text):
    lines = []
    in_ul = False
    for line in text.split('\n'):
        if line.startswith('### '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h4 style="color:#f1f5f9;margin:16px 0 4px;font-size:14px">{line[4:]}</h4>')
        elif line.startswith('## '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h3 style="color:#f1f5f9;margin:20px 0 6px;font-size:16px;border-bottom:1px solid #475569;padding-bottom:4px">{line[3:]}</h3>')
        elif line.startswith('# '):
            if in_ul: lines.append('</ul>'); in_ul = False
            lines.append(f'<h2 style="color:#e2e8f0;margin:24px 0 8px;font-size:18px">{line[2:]}</h2>')
        elif line.startswith('- ') or line.startswith('* '):
            if not in_ul: lines.append('<ul style="margin:4px 0 4px 16px;padding:0;color:#cbd5e1;font-size:13px">'); in_ul = True
            content = _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', line[2:])
            lines.append(f'<li style="margin:2px 0">{content}</li>')
        else:
            if in_ul: lines.append('</ul>'); in_ul = False
            if line.strip():
                content = _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', line)
                lines.append(f'<p style="color:#cbd5e1;font-size:13px;margin:4px 0;line-height:1.6">{content}</p>')
    if in_ul: lines.append('</ul>')
    return '\n'.join(lines)

_spin_out = widgets.Output()
_plan_html = widgets.HTML()
_cost_html = widgets.HTML()
_gap_html  = widgets.HTML()
_save_area = widgets.VBox([])
_prof_name = _state.get('profile', {}).get('name', 'Default')
display(ui.card_widget([
    widgets.HTML(f'<p style="color:#475569;font-size:11px;margin:0 0 8px">Profile: <b style="color:#94a3b8">{_prof_name}</b></p>'),
    _spin_out,
    _plan_html,
    _cost_html,
    _gap_html,
    _save_area,
], title='Generate my plan'))

with _spin_out:
    ui.progress('Generating your plan...')

_result = client.run_async('/generate', {
    'goal_type': _state['goal'],
    'locked_topics': _state.get('locked', {}),
    'facts': _state.get('profile', {}).get('facts', {}),
    'target': _state.get('target'),
}, output=_spin_out)

if not _result['ok']:
    with _spin_out:
        clear_output(wait=True)
    _plan_html.value = ui.error_html(_result.get('error', 'Generation failed'))
else:
    _data = _result['data']
    _plan_id = _data.get('plan_id', '')
    _content = _data.get('content', _data.get('plan', ''))
    _cost = _data.get('cost_usd', 0)
    _gaps = _data.get('gaps', [])
    _default_name = f"{_state['goal']} - {_date.today().isoformat()}"
    _plan_html.value = (
        '<div style="background:#1e293b;border:1px solid #475569;border-radius:8px;padding:20px;'
        'max-height:500px;overflow-y:auto;font-family:Inter,system-ui,sans-serif">'
        + _md_to_html(_content) + '</div>'
    )
    _cost_html.value = (
        f'<p style="color:#64748b;font-size:12px;margin:6px 0 0">'
        f'Cost: ${_cost:.4f} &nbsp;|&nbsp; {_data.get("tokens_used", 0):,} tokens</p>'
    )
    # -- Gap panel --
    if _gaps:
        _gap_rows = ''.join(
            '<div style="background:#1e293b;border:1px solid #334155;border-radius:6px;padding:10px 14px;margin:4px 0">'
            f'<p style="color:#f1f5f9;font-size:13px;font-weight:600;margin:0 0 2px">{_g["label"]}</p>'
            f'<p style="color:#64748b;font-size:11px;text-transform:uppercase;margin:0 0 6px">{_g["section_title"]}</p>'
            f'<p style="color:#94a3b8;font-size:12px;margin:0 0 6px">{_g["why_it_matters"]}</p>'
            f'<p style="color:#cbd5e1;font-size:12px;margin:0"><b style="color:#e2e8f0">How to answer: </b>{str(_g["how_to_satisfy"]).splitlines()[0]}</p>'
            '</div>'
            for _g in _gaps
        )
        _gap_html.value = (
            '<div style="background:#1e293b;border:1px solid #f59e0b;border-radius:6px;'
            'padding:12px 16px;margin:12px 0 8px">'
            f'<p style="color:#fbbf24;font-size:13px;font-weight:600;margin:0 0 4px">'
            f'\u26a0\ufe0f {len(_gaps)} required item(s) missing from this plan</p>'
            '<p style="color:#94a3b8;font-size:12px;margin:0">Your plan was written with what you provided. Filling these gaps will strengthen it.</p>'
            '</div>' + _gap_rows
        )

    # -- Save area --
    _hint_html = widgets.HTML(
        ui.instruction_html(['Refine with AI -- My Plans, Refine tab',
                             'Minor tweaks -- open in <a href="https://drive.google.com/drive/home" target="_blank" style="color:#3b82f6">Google Drive</a> directly',
                             'Export -- My Plans, Export tab (Executive Summary, Pitch Deck, and more)'])
    )
    _name_input = widgets.Text(
        value=_default_name,
        description='Name:',
        layout=widgets.Layout(width='420px'),
        style={'description_width': '50px'},
    )
    _save_btn = widgets.Button(description='Save to Drive', button_style='success',
                                layout=widgets.Layout(width='130px', height='34px'))
    _discard_btn = widgets.Button(description='Discard', button_style='warning',
                                   layout=widgets.Layout(width='90px', height='34px'))
    _save_status = widgets.HTML()

    def _on_save(btn):
        _save_btn.disabled = True
        _discard_btn.disabled = True
        _save_status.value = '<p style="color:#94a3b8;font-size:13px">Saving...</p>'
        try:
            name = _name_input.value.strip() or _default_name
            safe = name.replace('/', '-').replace('\\', '-')
            filename = f'{safe}.md'
            profile = _state.get('profile', {})
            folder = f"plans/{profile.get('id', 'default')}"
            meta = {
                'name': name,
                'goal_type': _state['goal'],
                'target': _state.get('target'),
                'locked_topics': _state.get('locked', {}),
                'gaps': _gaps,
                'profile_id': profile.get('id', 'default'),
                'plan_id': _plan_id,
                'cost_usd': _cost,
                'model': _data.get('model', ''),
                'version': '1',
                'created_at': _date.today().isoformat(),
            }
            drive_mgr.save(folder, filename, _content, meta=meta)
            _state['last_plan'] = {'name': name, 'plan_id': _plan_id, 'filename': filename}
            _name_input.disabled = True
            _save_status.value = (
                ui.success_html(f'Saved: {filename}')
                + ui.instruction_html('Scroll down to My Plans to export, refine, or start a new plan.')
            )
        except Exception as _e:
            _save_btn.disabled = False
            _discard_btn.disabled = False
            _save_status.value = ui.error_html(f'Save failed: {_e}')

    def _on_discard(btn):
        _name_input.disabled = True
        _save_btn.disabled = True
        _discard_btn.disabled = True
        _save_status.value = ui.warning_html('Discarded. Re-run to generate again.')

    _save_btn.on_click(_on_save)
    _discard_btn.on_click(_on_discard)
    _save_area.children = [
        _hint_html,
        widgets.HTML('<p style="color:#94a3b8;font-size:13px;margin:12px 0 6px">Save this plan?</p>'),
        widgets.HBox([_name_input, _save_btn, _discard_btn],
                      layout=widgets.Layout(align_items='center', gap='8px')),
        _save_status,
    ]


In [ ]:
# @title Account
import requests as _req
import ipywidgets as _w
from IPython.display import display as _display

result_acct = client.call('GET', '/account')
result_usage = client.get_usage()

sub_html = ''
if result_acct['ok']:
    d = result_acct['data']
    plan_label = d.get('plan', 'free').capitalize()
    limits = d.get('tier_limits', {})
    def _fmt(v):
        if v == 0: return 'Unlimited'
        if v == 'none': return 'Not included'
        return str(v)
    limits_rows = ''.join(f"<tr><td style='padding:3px 8px 3px 0;color:#94a3b8;text-transform:capitalize'>{k.replace('_', ' ')}</td><td style='padding:3px 0;color:#e2e8f0'>{_fmt(v)}</td></tr>" for k, v in limits.items())
    limits_html = f"<table style='border-collapse:collapse;margin-top:6px'>{limits_rows}</table>" if limits_rows else ''
    topup = f"Top-up balance: ${d.get('topup_balance_usd', 0):.2f}<br>" if d.get('topup_balance_usd') else ''
    sub_html = f"<b>{plan_label}</b><br>Status: {d.get('status', 'active')}<br>Monthly limit: ${d.get('monthly_limit_usd', 0):.2f}<br>{topup}<br><b>Plan limits</b>{limits_html}<br><a href='https://planforge.pysolvr.com/pricing' style='color:#60a5fa'>Manage plan</a>"
else:
    sub_html = f"<span style='color:#f87171'>{result_acct.get('error', 'Could not fetch account')}</span>"

usage_html = ''
if result_usage['ok']:
    data = result_usage['data']
    limit = result_acct['data'].get('monthly_limit_usd', 1) if result_acct['ok'] else 1
    usage_html = ui.usage_bar_html(data.get('current_month_spend_usd', 0), limit)
else:
    usage_html = f"<span style='color:#f87171'>{result_usage.get('error', 'Could not fetch usage')}</span>"

_current_ver = '0.5.0'
_latest_ver = _current_ver
try:
    _cl = _req.get('https://raw.githubusercontent.com/dAIvdMercer/planforge-client/main/CHANGELOG.md', timeout=5).text
    import re as _re
    _m = _re.search(r'## (\d+\.\d+\.\d+)', _cl)
    if _m: _latest_ver = _m.group(1)
except: pass

_cur_parts = tuple(int(x) for x in _current_ver.split('.'))
_lat_parts = tuple(int(x) for x in _latest_ver.split('.'))
if _lat_parts <= _cur_parts:
    _ver_status = '<span style="color:#10b981">Up to date</span>'
else:
    _ver_status = f'<span style="color:#f59e0b">Update available: v{_latest_ver}</span><br><small>Save a fresh copy from the link below</small>'
version_html = (f'<p><b>Current:</b> v{_current_ver}</p>'
    f'<p><b>Latest:</b> v{_latest_ver} -- {_ver_status}</p>'
    '<p><a href="https://github.com/dAIvdMercer/planforge-client/blob/main/CHANGELOG.md" style="color:#60a5fa">View changelog</a></p>')

rotate_html = (
    '<p>Generates a new API key and emails it to the address you subscribed with.</p>'
    '<p style="color:#f59e0b">Make sure you have access to that inbox before rotating -- your current key stops working immediately.</p>'
    '<button onclick="(function(){var r=confirm(\"Rotate your API key? Your current key will stop working immediately.\");if(r){IPython.notebook.kernel.execute(\"_rotate=client.call(\\\"POST\\\",\\\"/account/rotate-key\\\");print(_rotate.get(\\\"data\\\",{}).get(\\\"message\\\",_rotate.get(\\\"error\\\",\\\"Unknown error\\\")))\")}})()" style="background:#dc2626;color:white;border:none;padding:8px 16px;border-radius:4px;cursor:pointer;margin-top:8px">Rotate API key</button>')

# -- tab style --
_tab_style = _w.HTML(
    '<style>'
    '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab,'
    '.jupyter-widgets.widget-tab > .lm-TabBar .lm-TabBar-tab,'
    '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current,'
    '.jupyter-widgets.widget-tab > .lm-TabBar .lm-TabBar-tab.lm-mod-current,'
    '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab:hover,'
    '.jupyter-widgets.widget-tab > .lm-TabBar .lm-TabBar-tab:hover {'
    'background:transparent !important;color:#94a3b8;'
    'border-width:0 0 2px 0 !important;border-style:solid !important;'
    'border-color:transparent !important;box-shadow:none !important;'
    'font-size:13px;font-family:Inter,system-ui,sans-serif;padding:8px 16px;}'
    '.jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current,'
    '.jupyter-widgets.widget-tab > .lm-TabBar .lm-TabBar-tab.lm-mod-current {color:#f1f5f9 !important;border-bottom-color:#3B82F6 !important;}'
    '.jupyter-widgets.widget-tab > .p-TabBar,'
    '.jupyter-widgets.widget-tab > .lm-TabBar {border-bottom:1px solid #475569 !important;border-top:none !important;}'
    '.jupyter-widgets.widget-tab > .widget-tab-contents,'
    '.jupyter-widgets.widget-tab > .p-Widget.widget-tab-contents {'
    'border:none !important;padding:12px 0 0;background:transparent;}'
    '</style>'
)

# -- support tab widgets --
_type_dd = _w.Dropdown(
    options=[('Question', 'question'), ('Bug', 'bug'), ('Feature request', 'feature_request'), ('Feedback', 'feedback'), ('Billing', 'billing'), ('Data request', 'data_request')],
    description='Ticket type:',
    style={'description_width': 'initial'},
    layout=_w.Layout(width='320px'),
)
_text = _w.Textarea(placeholder='Describe your issue or feedback...', layout=_w.Layout(width='480px', height='100px'))
_btn = _w.Button(description='Submit ticket', button_style='primary')
_out = _w.Output()
def _on_submit(b):
    _out.clear_output()
    with _out:
        if not _text.value.strip():
            display(ui.warning_html('Please describe your issue before submitting.'))
            return
        result = client.call('POST', '/support', payload={
            'type': _type_dd.value,
            'feature_tag': 'planforge',
            'free_text': _text.value.strip(),
        })
        if result['ok']:
            display(ui.success_html(f"Ticket submitted. Reference: {result['data']['ticket_id']}"))
            _text.value = ''
        else:
            display(ui.error_html(result.get('error', 'Submission failed. Please try again.')))
_btn.on_click(_on_submit)

_support_static = _w.HTML('<p><b>Files:</b> Google Drive > pysolvr > planforge</p>'
    '<p><b>Docs:</b> <a href="https://planforge.pysolvr.com/docs" style="color:#60a5fa">planforge.pysolvr.com/docs</a></p>')

# -- build tabs --
_tabs = _w.Tab(children=[
    _w.VBox([_w.HTML(sub_html)]),
    _w.VBox([_w.HTML(usage_html)]),
    _w.VBox([_w.HTML(version_html)]),
    _w.VBox([_w.HTML(rotate_html)]),
    _w.VBox([_support_static, _w.HTML('<hr style="border-color:#475569;margin:12px 0">'), _type_dd, _text, _w.HBox([_btn], layout=_w.Layout(margin='6px 0 0')), _out]),
])
for _i, _t in enumerate(['Subscription', 'Usage', 'Version', 'Rotate Key', 'Support']):
    _tabs.set_title(_i, _t)

_display(ui.card_widget([_tab_style, _tabs]))
